# 서울시 체육시설 데이터 Selenium 크롤링

## 프로젝트 개요

서울열린데이터광장에서 제공하는 체육시설 공공데이터를
Selenium을 이용하여 동적으로 수집하고,
수집 결과를 CSV 파일로 중간 저장한다.

### 크롤링 대상
- 사이트: 서울열린데이터광장
- 검색어: 체육시설
- 데이터셋: 서울시 체육시설 공공서비스예약 정보

### 수집 항목
- 서비스구분
- 서비스ID
- 대분류명
- 소분류명
- 서비스상태
- 서비스명

In [4]:
import time

import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

## Selenium 실행 및 사이트 접속

Selenium WebDriver를 실행하고 서울열린데이터광장에 접속한다.

In [24]:
# Chrome WebDriver 실행

options = Options()
options.add_argument("--start-maximized")

driver = webdriver.Chrome(options=options)

wait = WebDriverWait(driver, 10)

# 서울열린데이터광장 접속
driver.get("https://data.seoul.go.kr/")

print('사이트 접속 완료')
print('현재 URL:', driver.current_url)
print('페이지 제목:', driver.title)

사이트 접속 완료
현재 URL: https://data.seoul.go.kr/
페이지 제목: 서울 열린데이터광장


## 데이터셋 검색 및 상세 페이지 접근

서울열린데이터광장에서 '체육시설'을 검색하고,
검색 결과에서 '서울시 체육시설 공공서비스예약 정보'
데이터셋의 상세 페이지로 이동한다.

In [23]:
# 공공데이터 페이지 접속

driver.get(
    'https://data.seoul.go.kr/dataList/datasetList.do'
)

wait.until(
    EC.presence_of_element_located(
        (By.ID, 'searchKeyword')
    )
)

print('공공데이터 페이지 접속 완료')

공공데이터 페이지 접속 완료


In [21]:
# 검색어 입력

search_input = wait.until(
    EC.element_to_be_clickable(
        (By.ID, 'searchKeyword')
    )
)

search_input.click()
search_input.clear()
search_input.send_keys('체육시설')

# 검색 버튼 클릭
search_button = wait.until(
    EC.element_to_be_clickable(
        (
            By.CSS_SELECTOR,
            "button.btn-ico[title='공공데이터 검색']"
        )
    )
)

search_button.click()

time.sleep(2)

print("검색 완료")

검색 완료


In [22]:
# 크롤링 대상 데이터셋 선택

target = wait.until(
    EC.element_to_be_clickable(
        (
            By.XPATH,
            "//a[contains(normalize-space(.), '서울시 체육시설 공공서비스예약 정보')]"
        )
    )
)

target.click()

wait.until(
    EC.presence_of_element_located(
        (By.ID, 'AXGridTarget_AX_gridBodyTable')
    )
)

time.sleep(2)

print('데이터셋 상세 페이지 접속 완료')
print('현재 URL:', driver.current_url)
print('페이지 제목:', driver.title)

데이터셋 상세 페이지 접속 완료
현재 URL: https://data.seoul.go.kr/dataList/OA-2266/S/1/datasetView.do
페이지 제목: 서울시 체육시설 공공서비스예약 정보> 데이터셋> 공공데이터 | 서울열린데이터광장


## 동적 데이터 크롤링

상세 페이지의 Sheet 영역에 동적으로 생성된 Grid 데이터를
Selenium을 이용하여 추출한다.

현재 데이터 Grid에서 실제 값이 존재하는
6개 필드를 수집한다.

In [18]:
# 동적으로 생성된 데이터 Grid 찾기

data_table = wait.until(
    EC.presence_of_element_located(
        (By.ID, 'AXGridTarget_AX_gridBodyTable')
    )
)

# Grid의 데이터 행 가져오기
rows = data_table.find_elements(
    By.CSS_SELECTOR,
    'tbody#AXGridTarget_AX_tbody tr'
)

columns = [
    '서비스구분',
    '서비스ID',
    '대분류명',
    '소분류명',
    '서비스상태',
    '서비스명'
]

data = []

for row in rows:
    cells = row.find_elements(By.TAG_NAME, 'td')

    if len(cells) < 6:
        continue

    values = [
        cells[i].text.strip()
        for i in range(6)
    ]

    # 실제 데이터 행만 수집
    if values[1]:
        data.append(values)

print('수집 완료')
print('수집 데이터:', len(data), '건')

수집 완료
수집 데이터: 18 건


In [17]:
# 수집 데이터를 DataFrame으로 변환

df = pd.DataFrame(
    data,
    columns=columns
)

print('DataFrame 크기:', df.shape)

DataFrame 크기: (18, 6)


In [13]:
display(df)

,서비스구분,서비스ID,대분류명,소분류명,서비스상태,서비스명
0,자체,S251121100349891778,체육시설,테니스장,접수중,테니스장1(평일)-2026년 응봉공원(대현산배수지)
1,자체,S251121102355938084,체육시설,테니스장,접수중,테니스장1(토/일/공휴일)-2026년 응봉공원(대현산배수지)
2,자체,S251121103012746640,체육시설,테니스장,접수중,테니스장2(평일)-2026년 응봉공원(대현산배수지)
3,자체,S251121103347587479,체육시설,테니스장,접수중,테니스장2(토/일/공휴일)-2026년 응봉공원(대현산배수지)
4,자체,S251121103722953401,체육시설,다목적경기장,접수중,다목적구장-2026년 응봉공원(대현산배수지)
5,자체,S210401100008601453,체육시설,축구장,접수종료,○ [평일] (주간) 마포 난지천 인조잔디축구장
6,자체,S260706132614891677,체육시설,풋살장,접수중,8월 [평일]월드컵난지천공원 풋살장 대여 - 7월15일(수) 오후1시 접수
7,자체,S260706134128763150,체육시설,풋살장,접수중,8월 [공휴일주말]월드컵난지천공원 풋살장 대여 - 7월15일(수) 오후1시 접수
8,자체,S241210095552925088,체육시설,풋살장,접수종료,잠실제3풋살경기장-토/일/공휴일(야간사용)
9,자체,S241210100011131040,체육시설,풋살장,접수종료,잠실제3풋살경기장-토/일/공휴일(주간사용)


## 크롤링 결과 중간 저장

Selenium으로 수집한 데이터를 Pandas DataFrame으로 변환한 후
CSV 파일로 저장한다.

이 CSV 파일은 다음 단계인 데이터 전처리에서 사용한다.

In [16]:
# 중간 저장

output_file = 'sports_crawling_raw.csv'

df.to_csv(
    output_file,
    index=False,
    encoding='utf-8-sig'
)

print('중간 저장 완료')
print('파일명:', output_file)
print('저장 데이터:', len(df), '건')

중간 저장 완료
파일명: sports_crawling_raw.csv
저장 데이터: 18 건


In [25]:
# Selenium 종료

driver.quit()

print('웹페이지 종료')

웹페이지 종료
